In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

data = [
    # SALES_DOC_NUMBER, SALES_DOC_ITEMNUMBER, updated_ts,         created_ts,          other_cols
    ("SO1",             "10",                 "2024-08-01 09:00", "2024-07-31 18:00",  "A1"),
    ("SO1",             "10",                 "2024-08-02 10:15", "2024-08-01 09:05",  "A2"),  # newest updated_ts → should WIN for (SO1,10)
    ("SO1",             "20",                 "2024-08-01 08:00", "2024-07-30 11:00",  "B1"),
    ("SO1",             "20",                 "2024-08-01 08:00", "2024-07-31 10:00",  "B2"),  # same updated_ts as above, but newer created_ts → should WIN for (SO1,20)
    ("SO2",             "10",                 "2024-08-03 14:30", "2024-08-02 16:00",  "C1"),  # only one row in group → kept
]

df = spark.createDataFrame(data, ["SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER", "updated_ts", "created_ts", "payload"])
display(df)

# Convert the timestamp strings into proper timestamp type (recommended!)
df = (df.withColumn("updated_ts", to_timestamp("updated_ts")).withColumn("created_ts", to_timestamp("created_ts")))

print("=== Input Data ===")
# df.show(truncate=False)
display(df)



In [0]:
df1 = df.orderBy(col("updated_ts").desc())
display(df1)

In [0]:

df2 = df.orderBy(
    col("updated_ts").desc(),
    col("created_ts").desc()
)
print("=== Step 2: Sorted by updated_ts DESC, then created_ts DESC ===")
display(df2)


In [0]:

print("=== Step 3: Group by SALES_DOC_NUMBER, SALES_DOC_ITEMNUMBER ===")
display(
    df.orderBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER", 
               col("updated_ts").desc(), col("created_ts").desc())
)


In [0]:

w = Window.partitionBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER") \
          .orderBy(col("updated_ts").desc(), col("created_ts").desc())
          
w2 = Window.partitionBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER")
display(w2)
df_with_rank = df.withColumn("latestTS", row_number().over(w))

print("=== Step 4: Window applied (row_number inside each group) ===")
display(df_with_rank)


In [0]:

result = df_with_rank.where(col("latestTS") == 1).drop("latestTS")

print("=== Step 5: Final deduplicated output ===")
display(result)


In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

data = [
    ("SO1", "10", "2024-08-01 09:00", "2024-07-31 18:00", "A1"),
    ("SO1", "10", "2024-08-02 10:15", "2024-08-01 09:05", "A2"),
    ("SO1", "20", "2024-08-01 08:00", "2024-07-30 11:00", "B1"),
    ("SO1", "20", "2024-08-01 08:00", "2024-07-31 10:00", "B2"),
    ("SO2", "10", "2024-08-03 14:30", "2024-08-02 16:00", "C1"),
]

df = spark.createDataFrame(data, 
    ["SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER", "updated_ts", "created_ts", "payload"])

df = df.withColumn("updated_ts", to_timestamp("updated_ts")) \
       .withColumn("created_ts", to_timestamp("created_ts"))

print("=== Step 0: Input Data ===")
display(df)

# Step 1: Show ordering by updated_ts
df1 = df.orderBy(col("updated_ts").desc())
print("=== Step 1: Sorted by updated_ts DESC ===")
display(df1)

# Step 2: Show ordering by updated_ts, created_ts
df2 = df.orderBy(col("updated_ts").desc(), col("created_ts").desc())
print("=== Step 2: Sorted by updated_ts DESC, created_ts DESC ===")
display(df2)

# Step 3: Show grouping visually
df3 = df.orderBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER",
                 col("updated_ts").desc(), col("created_ts").desc())
print("=== Step 3: Grouping View (manual order inside groups) ===")
display(df3)

# Step 4: Apply window
w = Window.partitionBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER") \
          .orderBy(col("updated_ts").desc(), col("created_ts").desc())

df4 = df.withColumn("latestTS", row_number().over(w))
print("=== Step 4: Window row_number inside each group ===")
display(df4)

# Step 5: Filter final latest rows
result = df4.where(col("latestTS") == 1).drop("latestTS")
print("=== Step 5: Final Output ===")
display(result)


In [0]:

w = Window.partitionBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER") \
          .orderBy(col("updated_ts").desc(), col("created_ts").desc())

result = (df
    .withColumn("latestTS", row_number().over(w))
    .where(col("latestTS") == 1)    # keep the top row in each group
    .drop("latestTS")
)

print("=== Result: Latest row per (SALES_DOC_NUMBER, SALES_DOC_ITEMNUMBER) ===")
result.show(truncate=False)



In [0]:

debug = df.withColumn("rank_in_grp", row_number().over(w))
print("=== Row-number within each group (before filtering) ===")
debug.orderBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER", col("rank_in_grp")).show(truncate=False)


In [0]:

w_tiebreak = Window.partitionBy("SALES_DOC_NUMBER", "SALES_DOC_ITEMNUMBER") \
                   .orderBy(
                       col("updated_ts").desc(),
                       col("created_ts").desc(),
                       col("SAP_ADJ_REQ_ID").desc()  # ← extra tie-breaker if available
                   )

result_stable = (df
    .withColumn("latestTS", row_number().over(w_tiebreak))
    .where(col("latestTS") == 1)
    .drop("latestTS")
)


SCD type 2 using PySpark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_date, when, lower, trim
from pyspark.sql.types import DateType

# Create a Spark session
spark = SparkSession.builder \
    .appName("SCD Type 2 Implementation") \
    .getOrCreate()

# Load staging data
staging_data = [
    (1, 'John Doe', '123 Elm St'),
    (2, 'Jane Smith', '456 Oak St'),
    (3, 'Jim Brown', '789 Pine St')
]

staging_df = spark.createDataFrame(staging_data, ["customer_id", "customer_name", "address"])

# Load final table data
final_data = [
    (1, 'John Doe', '123 Maple St', '2024-01-01', '9999-12-31', 1),
    (2, 'Jane Smith', '456 Oak St', '2024-02-01', '9999-12-31', 1)
]

final_df = spark.createDataFrame(final_data, ["customer_id", "customer_name", "address", "effective_date", "expiration_date", "current_flag"])

display(staging_df)
display(final_df)

In [0]:
# Step 1: Insert New Records
new_records_df = staging_df.join(final_df, on="customer_id", how="left_anti") \
    .withColumn("effective_date", current_date()) \
    .withColumn("expiration_date", lit("9999-12-31").cast(DateType())) \
    .withColumn("current_flag", lit(1))

In [0]:
# Step 2: Expire Existing Records
expired_records_df = final_df.alias("f").join(
    staging_df.alias("s"), on="customer_id", how="inner"
).where(
    (lower(trim(col("f.customer_name"))) != lower(trim(col("s.customer_name")))) | 
    (lower(trim(col("f.address"))) != lower(trim(col("s.address"))))
).select(
    col("f.customer_id"),
    col("f.customer_name"),
    col("f.address"),
    col("f.effective_date"),
    current_date().alias("expiration_date"),  # Set expiration date to current date
    lit(0).alias("current_flag")  # Set current flag to 0
)

# Update the final table with expired records
updated_final_df = final_df.alias("f").join(
    expired_records_df.select("customer_id", "expiration_date").alias("e"), on="customer_id", how="left"
).select(
    col("f.customer_id"),
    col("f.customer_name"),
    col("f.address"),
    col("f.effective_date"),
    when(col("e.expiration_date").isNotNull(), col("e.expiration_date")).otherwise(col("f.expiration_date")).alias("expiration_date"),
    when(col("e.expiration_date").isNotNull(), lit(0)).otherwise(col("f.current_flag")).alias("current_flag")
)

In [0]:
# Step 3: Insert Updated Records
updated_records_df = staging_df.alias("s").join(
    final_df.alias("f"), on="customer_id", how="inner"
).where(
    (lower(trim(col("f.customer_name"))) != lower(trim(col("s.customer_name")))) | 
    (lower(trim(col("f.address"))) != lower(trim(col("s.address"))))
).select(
    col("s.customer_id"),
    col("s.customer_name"),
    col("s.address"),
    current_date().alias("effective_date"),  # Set new effective date to current date
    lit("9999-12-31").cast(DateType()).alias("expiration_date"),  # Default future date
    lit(1).alias("current_flag")  # Set current flag to 1
)

# Combine all changes into the final dataframe
result_df = updated_final_df.union(new_records_df).union(updated_records_df)

# Show final result
result_df.show()

----------------------------SCD 2 COMPLETE--------------------------------